In [129]:
import pandas as pd
import ast
import json
#from notebooks.jh.preprocessing.ipynb.normalize.common import *
#from notebooks.jh.preprocessing.load_tree import *
import re

In [130]:
with open("/app/data/raw/prod_info.json", 'r') as file:
    meta_data = json.load(file)

meta_data = pd.json_normalize(meta_data)

In [131]:
customer_info_columns = [nm for nm in list(meta_data.columns) if 'customerInfo' in nm and nm.endswith("valueList")]
customer_info_columns


['customerInfo.onboardingCustomer.customerTypeRule.valueList',
 'customerInfo.onboardingCustomer.businessCustomerSubtypeRule.valueList',
 'customerInfo.onboardingCustomer.individualCustomerSubtypeRule.valueList']

In [132]:
# NaN을 빈 리스트로 변환
for col in customer_info_columns:
    meta_data[col] = meta_data[col].apply(lambda x: x if isinstance(x, list) else [])

# 각 컬럼별로 유니크 set 구하기 (NaN 제거)
unique_sets = {}
for col in customer_info_columns:
    exploded = meta_data.explode(col)
    # NaN, None, '' 모두 제거
    filtered = exploded[col][pd.notna(exploded[col])]
    unique_set = set(filtered) - {''}
    unique_sets[col.replace('.', '|').lower()] = unique_set

In [133]:
rdb_path = "/app/data/rdb_prepro/meta.csv"

# 전처리 

In [134]:
df = pd.read_csv(rdb_path)

In [135]:
def parse_age_range(age_criteria_str):
    min_age, max_age = 0, 150  # 기본값으로 min_age는 0, max_age는 150으로 설정
    try:
        age_criteria = ast.literal_eval(age_criteria_str)
        # 리스트 내에서 나이 정보를 추출
        for item in age_criteria:
            if '이상' in item:
                min_age = int(re.search(r'\d+', item).group())
            elif '이하' in item:
                max_age = int(re.search(r'\d+', item).group())
    except (ValueError, SyntaxError):
        pd.Series([0, 150])
    return pd.Series([min_age, max_age])

In [136]:
def parse_age_range(age_criteria_str):
    min_age, max_age = 0, 150  # 기본값으로 min_age는 0, max_age는 150으로 설정
    try:
        age_criteria = ast.literal_eval(age_criteria_str)
        # 리스트 내에서 나이 정보를 추출
        for item in age_criteria:
            if '이상' in item:
                min_age = int(re.search(r'\d+', item).group())
            elif '이하' in item:
                max_age = int(re.search(r'\d+', item).group())
    except (ValueError, SyntaxError):
        pd.Series([0, 150])
    return pd.Series([min_age, max_age])

In [137]:
description_df = df.copy()
col_value = 'customerinfo|onboardingcustomer|agerule|value'
col_elig = 'customerinfo|onboardingcustomer|agerule|eligibility'
if col_value in df.columns and col_elig in df.columns:
   description_df[['minAge', 'maxAge']] = df[col_value].apply(parse_age_range)
   description_df.drop(columns=[col_elig], inplace=True)

In [138]:
description_df.loc[description_df.pmProductId=="PA00000068"]

,pmProductId,customerinfo|onboardingcustomerwelfaretype|welfarediscounttypeavailability,customerinfo|onboardingcustomer|agerule|value,customerinfo|onboardingcustomer|businesscustomersubtyperule|eligibility,customerinfo|onboardingcustomer|businesscustomersubtyperule|valuelist,customerinfo|onboardingcustomer|customertyperule|eligibility,customerinfo|onboardingcustomer|customertyperule|valuelist,customerinfo|onboardingcustomer|individualcustomersubtyperule|eligibility,customerinfo|onboardingcustomer|individualcustomersubtyperule|valuelist,customerinfo|onboardingcustomer|organizationcustomersubtyperule,minAge,maxAge
67,PA00000068,{},"['연령 (일기준) 나이 이상 19', '연령 (월기준) 나이 이하 34']",NaN,NaN,NaN,NaN,True,"['공무원', '일반']",{},19,34


In [139]:
def preprocessing_eligibility(row, col_value, col_elig, unique_set:set):
    """ eligibility: True 기준으로 맞추는 함수 """
    values = row[col_value]
    eligibility = row[col_elig]

    if pd.isna(values) or values == '정보없음':
        return "제한 없음"
    
    if not isinstance(values, list):
        try:
            values = ast.literal_eval(values)
        except:
            if values.lower() =='all' and (eligibility == "True" or eligibility == True):
                return f"개인 가입 가능"
            elif values.lower() =='all' and (eligibility == "False" or eligibility == False):
                unique_set.discard("개인")
                return f"{unique_set} 가입 가능"
            else:
                values = [values]
    
    # eligibility 처리
    if eligibility is True or eligibility == "True":   
        return f"{values} 가입 가능"
    
    elif eligibility is False or eligibility == "False":
        for value in values:
            unique_set.discard(value)
        values = list(unique_set)
        return f"{values} 가입 가능"
    else:
        raise ValueError(f"eligibility: {eligibility}, col_elig: {col_elig}")
        
    

In [140]:
col_value = 'customerinfo|onboardingcustomer|businesscustomersubtyperule|valuelist'
col_elig = 'customerinfo|onboardingcustomer|businesscustomersubtyperule|eligibility'
if col_value in df.columns and col_elig in df.columns:
   description_df[col_value] = df.apply(preprocessing_eligibility, axis=1, args=(col_value, col_elig, unique_sets[col_value.lower()]))
   description_df.drop(columns=[col_elig], inplace=True)

In [141]:
col_value = 'customerinfo|onboardingcustomer|customertyperule|valuelist'
col_elig = 'customerinfo|onboardingcustomer|customertyperule|eligibility'
if col_value in df.columns and col_elig in df.columns:
   description_df[col_value] = df.apply(preprocessing_eligibility, axis=1, args=(col_value, col_elig, unique_sets[col_value.lower()]))
   description_df.drop(columns=[col_elig], inplace=True)

In [142]:
col_value = 'customerinfo|onboardingcustomer|individualcustomersubtyperule|valuelist'
col_elig = 'customerinfo|onboardingcustomer|individualcustomersubtyperule|eligibility'
if col_value in df.columns and col_elig in df.columns:
   description_df[col_value] = df.apply(preprocessing_eligibility, axis=1, args=(col_value, col_elig, unique_sets[col_value.lower()]))
   description_df.drop(columns=[col_elig], inplace=True)

In [143]:
description_df.drop(columns=['customerinfo|onboardingcustomerwelfaretype|welfarediscounttypeavailability', 'customerinfo|onboardingcustomer|organizationcustomersubtyperule'], inplace=True)

In [144]:
description_df.head(3)

,pmProductId,customerinfo|onboardingcustomer|agerule|value,customerinfo|onboardingcustomer|businesscustomersubtyperule|valuelist,customerinfo|onboardingcustomer|customertyperule|valuelist,customerinfo|onboardingcustomer|individualcustomersubtyperule|valuelist,minAge,maxAge
0,PA00000001,NaN,제한 없음,제한 없음,제한 없음,0,150
1,PA00000002,NaN,제한 없음,제한 없음,제한 없음,0,150
2,PA00000003,NaN,제한 없음,제한 없음,제한 없음,0,150


In [17]:
column_list = list(description_df.columns)
description_dict = {}
for column in column_list:
    traverser.get_inherited_metadata(column)
    description_dict[column] = traverser.get_inherited_metadata(column)

In [18]:
documents, column_description_dict, column_mapper = generate_documents_and_mappings(description_df, description_dict)

In [19]:
save_mappings(
    column_description_dict, 
    column_mapper,
    "/app/data/prod_meta/rdb_reformulate/customerinfo/description.json",
    "/app/data/prod_meta/rdb_reformulate/customerinfo/column_mapper.json"
)

In [20]:
documents_df = documents_to_dataframe(documents)

In [21]:
documents_df.to_csv("/app/data/prod_meta/rdb_reformulate/customerinfo/documents.csv", index=False, encoding="utf-8-sig")

# 검색용 파싱

In [173]:
search_df = df.copy()

In [180]:
def parse_age_range(age_criteria_str):
    min_age, max_age = 0, 150  # 기본값으로 min_age는 0, max_age는 150으로 설정
    try:
        age_criteria = ast.literal_eval(age_criteria_str)
    except:
        age_criteria = [age_criteria_str]
        # 리스트 내에서 나이 정보를 추출
    
    try:    
        for item in age_criteria:
            if '이상' in item:
                min_age = int(re.search(r'\d+', item).group())
            elif '이하' in item:
                max_age = int(re.search(r'\d+', item).group())
    except Exception as e:
        print(e)
        print(age_criteria_str)
        return [0,150]
        
    return [min_age, max_age]

SyntaxError: default 'except:' must be last (1194389815.py, line 5)

In [181]:
col_value = 'customerinfo|onboardingcustomer|agerule|value'
col_elig = 'customerinfo|onboardingcustomer|agerule|eligibility'
if col_value in df.columns and col_elig in df.columns:
   search_df['customerinfo|onboardingcustomer|agerule|value'] = df[col_value].apply(parse_age_range)
   search_df.drop(columns=[col_elig], inplace=True)

malformed node or string: nan
nan
malformed node or string: nan
nan
malformed node or string: nan
nan
malformed node or string: nan
nan
malformed node or string: nan
nan
malformed node or string: nan
nan
malformed node or string: nan
nan
malformed node or string: nan
nan
malformed node or string: nan
nan
malformed node or string: nan
nan
malformed node or string: nan
nan
malformed node or string: nan
nan
malformed node or string: nan
nan
malformed node or string: nan
nan
invalid syntax (<unknown>, line 1)
연령 (월기준) 나이 이하 34
invalid syntax (<unknown>, line 1)
연령 (일기준) 나이 이상 65
invalid syntax (<unknown>, line 1)
연령 (일기준) 나이 이상 65
invalid syntax (<unknown>, line 1)
연령 (월기준) 나이 이하 34
invalid syntax (<unknown>, line 1)
연령 (월기준) 나이 이하 34
invalid syntax (<unknown>, line 1)
연령 (월기준) 나이 이하 34
invalid syntax (<unknown>, line 1)
연령 (월기준) 나이 이하 34
malformed node or string: nan
nan
malformed node or string: nan
nan
malformed node or string: nan
nan
malformed node or string: nan
nan
malformed node or

KeyError: "['customerinfo|onboardingcustomer|agerule|eligibility'] not found in axis"

In [161]:
def preprocessing_eligibility(row, col_value, col_elig, unique_set:set):
    """ eligibility: True 기준으로 맞추는 함수 """
    values = row[col_value]
    eligibility = row[col_elig]

    if pd.isna(values) or values == '정보없음' or not values:
        return list(unique_set)
    
    if not isinstance(values, list):
        try:
            values = ast.literal_eval(values)
        except:
            if values.lower() =='all' and (eligibility == "True" or eligibility == True):
                return f"개인 가입 가능"
            elif values.lower() =='all' and (eligibility == "False" or eligibility == False):
                unique_set.discard("개인")
                return f"{unique_set} 가입 가능"
            else:
                values = [values]
    
    # eligibility 처리
    if eligibility is True or eligibility == "True":   
        return values
    
    elif eligibility is False or eligibility == "False":
        for value in values:
            unique_set.discard(value)
        values = list(unique_set)
        return values
    else:
        raise ValueError(f"eligibility: {eligibility}, col_elig: {col_elig}")
        
    

In [162]:
col_value = 'customerinfo|onboardingcustomer|businesscustomersubtyperule|valuelist'
col_elig = 'customerinfo|onboardingcustomer|businesscustomersubtyperule|eligibility'
if col_value in df.columns and col_elig in df.columns:
   search_df[col_value] = df.apply(preprocessing_eligibility, axis=1, args=(col_value, col_elig, unique_sets[col_value.lower()]))
   search_df.drop(columns=[col_elig], inplace=True)

In [163]:
col_value = 'customerinfo|onboardingcustomer|customertyperule|valuelist'
col_elig = 'customerinfo|onboardingcustomer|customertyperule|eligibility'
if col_value in df.columns and col_elig in df.columns:
   search_df[col_value] = df.apply(preprocessing_eligibility, axis=1, args=(col_value, col_elig, unique_sets[col_value.lower()]))
   search_df.drop(columns=[col_elig], inplace=True)

In [164]:
col_value = 'customerinfo|onboardingcustomer|individualcustomersubtyperule|valuelist'
col_elig = 'customerinfo|onboardingcustomer|individualcustomersubtyperule|eligibility'
if col_value in df.columns and col_elig in df.columns:
   search_df[col_value] = df.apply(preprocessing_eligibility, axis=1, args=(col_value, col_elig, unique_sets[col_value.lower()]))
   search_df.drop(columns=[col_elig], inplace=True)

In [165]:
column_mapper = json.load(open("/app/data/rdb_reformulate/customerinfo/column_mapper.json"))

In [166]:
search_df.rename(columns=column_mapper, inplace=True)

In [167]:
#search_df.to_csv("/app/data/rdb_reformulate/customerinfo/meta2.csv", index=False, encoding="utf-8-sig")

In [170]:
pd.set_option('display.max_rows', 1000)      # or None for unlimited
pd.set_option('display.max_columns', 1000)   # or None for unlimited
search_df.loc[search_df.pmProductId=="PA00000106"]

,pmProductId,welfarediscounttypeavailability,agerule,businesscustomersubtyperule,customertyperule,individualcustomersubtyperule,organizationcustomersubtyperule
105,PA00000106,{},"[0, 150]","[SKTelecom㈜, SKBroadband㈜]",[개인],"[공무원, 일반]",{}


In [171]:
df.loc[df.pmProductId=="PA00000106"]

,pmProductId,customerinfo|onboardingcustomerwelfaretype|welfarediscounttypeavailability,customerinfo|onboardingcustomer|agerule|eligibility,customerinfo|onboardingcustomer|agerule|value,customerinfo|onboardingcustomer|businesscustomersubtyperule|eligibility,customerinfo|onboardingcustomer|businesscustomersubtyperule|valuelist,customerinfo|onboardingcustomer|customertyperule|eligibility,customerinfo|onboardingcustomer|customertyperule|valuelist,customerinfo|onboardingcustomer|individualcustomersubtyperule|eligibility,customerinfo|onboardingcustomer|individualcustomersubtyperule|valuelist,customerinfo|onboardingcustomer|organizationcustomersubtyperule
105,PA00000106,{},True,연령 (월기준) 나이 이하 34,NaN,NaN,True,개인,NaN,NaN,{}
